<a href="https://colab.research.google.com/github/bokofod/RothC_SOC/blob/main/Mongolia_Species_Suitability_Tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Species Suitability Rasters

**Earthbanc** | Google Earth Engine via Python API

Port of `scripts/Parameter_Species_Suitability_Tools-<region>-export.js` family
to Python. Generates **three** raster types per species at **30 m**, exported
as GeoTIFF to Google Drive:

- **Suitability**     - 0 = Unsuitable, 1 = Marginal, 2 = Optimal
- **FailureCount**    - count of *enabled* parameters with score = 0
- **LimitingFactor**  - single code 1-15 naming the dominant limit (legend in Cell 6 docstring)

### Supported regions (set `REGION` in Cell 3)

| Region            | Species | Source                                              | Notes                              |
|-------------------|--------:|-----------------------------------------------------|------------------------------------|
| `Mongolia`        |      20 | FAO GAUL country boundary                           | Arid; precip params disabled       |
| `ArunachalPradesh`|      15 | EB India ADM3-5 `stname='ARUNACHAL PRADESH'`        | Mountainous; slope cap 70 %        |
| `Rajasthan`       |      22 | EB India ADM3-5 `stname='RAJASTHAN'`                | Semi-arid; precip floor 100 mm     |
| `Sundarbans`      |      64 | EB India ADM3-5 `stname='WEST BENGAL'` + bbox       | Mangrove delta; flat (slope 30 %)  |
| `Uttarakhand`     |       7 | EB India ADM3-5 `stname='UTTARAKHAND'`              | Himalayan; slope cap 70 %          |

---

### How to use
1. Run **Cell 2** - install + GEE auth (once per session)
2. Edit **Cell 3** - set `REGION`, optionally tweak `PARAMETER_SETTINGS`/`MANAGEMENT`/`PRECIP_SOURCE`
3. Run **Cells 4-7** in order
4. Re-run **Cell 8** any time to refresh task status

Cell 7 is **idempotent across sessions**: it skips exports whose name (timestamp-stripped)
matches an existing useful task (COMPLETED / RUNNING / READY). Set `SPECIES_START_FROM`
in Cell 3 to skip species below a given letter (alphabetical), e.g. for resuming a partial run.


In [ ]:
# ============================================================
# CELL 2 - Install dependencies and authenticate
# Run once per Colab session. You will be prompted to log in
# with your Google account that has GEE access.
# ============================================================

!pip install earthengine-api --quiet

import ee

ee.Authenticate()
ee.Initialize(project='plus-earthbanc-io')

print('Earth Engine initialised')


In [ ]:
# ============================================================
# CELL 3 - Configuration
# Edit values here; downstream cells read from these globals.
# ============================================================

from datetime import datetime

# ── REGION SELECTOR ─────────────────────────────────────────
# Available: Mongolia | ArunachalPradesh | Rajasthan | Sundarbans | Uttarakhand
REGION = 'Mongolia'

REGION_CONFIG = {
    'Mongolia': {
        'asset':                 'FAO/GAUL/2015/level0',
        'asset_filter':          ('ADM0_NAME', 'Mongolia'),
        'bbox_clip':             None,
        'scope_min_precip_mm':   0,
        'scope_max_slope_pct':   50,
        'esa_classes':           [10, 20, 30, 40],
        'disable_precip_params': True,
        'note':                  'Arid country; precip params disabled, precip floor 0.',
    },
    'ArunachalPradesh': {
        'asset':                 'projects/plus-earthbanc-io/assets/EB/India/India_Block_ADM3_5',
        'asset_filter':          ('stname', 'ARUNACHAL PRADESH'),
        'bbox_clip':             None,
        'scope_min_precip_mm':   200,
        'scope_max_slope_pct':   70,
        'esa_classes':           [10, 20, 30, 40],
        'disable_precip_params': False,
        'note':                  'NE India, high rainfall, mountainous (slope cap 70%).',
    },
    'Rajasthan': {
        'asset':                 'projects/plus-earthbanc-io/assets/EB/India/India_Block_ADM3_5',
        'asset_filter':          ('stname', 'RAJASTHAN'),
        'bbox_clip':             None,
        'scope_min_precip_mm':   100,
        'scope_max_slope_pct':   50,
        'esa_classes':           [10, 20, 30, 40, 60],   # incl. bare/desert
        'disable_precip_params': False,
        'note':                  'Semi-arid; lower precip floor, includes ESA bare class.',
    },
    'Sundarbans': {
        'asset':                 'projects/plus-earthbanc-io/assets/EB/India/India_Block_ADM3_5',
        'asset_filter':          ('stname', 'WEST BENGAL'),
        'bbox_clip':             [88.0, 21.5, 89.5, 22.5],
        'scope_min_precip_mm':   200,
        'scope_max_slope_pct':   30,
        'esa_classes':           [10, 20, 30, 40, 90, 95],   # incl. wetland/mangrove
        'disable_precip_params': False,
        'note':                  'Indian Sundarbans mangrove delta. WB filter + bbox clip; includes ESA mangrove.',
    },
    'Uttarakhand': {
        'asset':                 'projects/plus-earthbanc-io/assets/EB/India/India_Block_ADM3_5',
        'asset_filter':          ('stname', 'UTTARAKHAND'),
        'bbox_clip':             None,
        'scope_min_precip_mm':   200,
        'scope_max_slope_pct':   70,
        'esa_classes':           [10, 20, 30, 40],
        'disable_precip_params': False,
        'note':                  'Himalayan; slope cap 70%.',
    },
}

assert REGION in REGION_CONFIG, f"REGION must be one of {list(REGION_CONFIG)}, got {REGION!r}"
cfg = REGION_CONFIG[REGION]

# ── Derived from REGION ─────────────────────────────────────
DRIVE_FOLDER          = f'SpeciesSuitabilityExports_{REGION}'
EXPORT_REGION_TAG     = REGION + '_'
SCOPE_INCLUDE_CLASSES = cfg['esa_classes']
SCOPE_MAX_SLOPE_PCT   = cfg['scope_max_slope_pct']
SCOPE_MIN_PRECIP_MM   = cfg['scope_min_precip_mm']

# ── Universal settings ──────────────────────────────────────
EXPORT_SCALE_M = 30
TIMESTAMP      = datetime.now().strftime('%Y%m%d_%H%M')

# Per-parameter enable + hard/soft flag (JS line 3489 defaults)
PARAMETER_SETTINGS = {
    'altitude':          {'enabled': True, 'hard': True},
    'frost':             {'enabled': True, 'hard': True},
    'maxTemp':           {'enabled': True, 'hard': True},
    'pH':                {'enabled': True, 'hard': False},
    'rainfall':          {'enabled': True, 'hard': False},
    'slope':             {'enabled': True, 'hard': False},
    'aspect':            {'enabled': True, 'hard': False},
    'soil':              {'enabled': True, 'hard': False},
    'precipWettest':     {'enabled': True, 'hard': False},
    'precipSeasonality': {'enabled': True, 'hard': False},
}

# Apply region-specific overrides (e.g. Mongolia disables all precip params)
if cfg.get('disable_precip_params'):
    for p in ('rainfall', 'precipWettest', 'precipSeasonality'):
        PARAMETER_SETTINGS[p]['enabled'] = False

# Management interventions (all off = pure natural regeneration baseline)
MANAGEMENT = {
    'microclimate':  False,
    'soilAmendment': False,
    'irrigation':    False,
    'drainage':      False,
    'terracing':     False,
}

# Ecology / classification mode
ECOLOGY_TYPE  = 'intermediate'   # generalist | intermediate | specialist
CAUTIOUS_MODE = False
ECOLOGY_THRESHOLDS = {
    'generalist':   {'optimalThreshold': 0.60, 'maxUnmanagedMarginals': 4},
    'intermediate': {'optimalThreshold': 0.70, 'maxUnmanagedMarginals': 3},
    'specialist':   {'optimalThreshold': 0.85, 'maxUnmanagedMarginals': 1},
}

# Precipitation source (used even when precip params are disabled - other code reads it)
PRECIP_SOURCE = 'CHIRPS'   # CHIRPS | WORLDCLIM | PROJECTED

# Species filter — alphabetical from this letter onwards (inclusive, case-insensitive).
# Useful for resuming after a partial run. '' = process every species.
SPECIES_START_FROM = ''

# ── Summary ─────────────────────────────────────────────────
print('Configuration set')
print(f'  Region            : {REGION}')
print(f'                      ({cfg["note"]})')
print(f'  Drive folder      : {DRIVE_FOLDER}')
print(f'  Export scale      : {EXPORT_SCALE_M} m')
print(f'  Timestamp         : {TIMESTAMP}')
print(f'  Scope precip floor: {SCOPE_MIN_PRECIP_MM} mm/yr' + (' (disabled)' if SCOPE_MIN_PRECIP_MM == 0 else ''))
print(f'  Scope slope max   : {SCOPE_MAX_SLOPE_PCT}%')
print(f'  ESA classes       : {SCOPE_INCLUDE_CLASSES}')
print(f'  Ecology           : {ECOLOGY_TYPE} | cautious: {CAUTIOUS_MODE}')
print(f'  Precipitation     : {PRECIP_SOURCE}')
print(f'  Disabled params   : {[k for k,v in PARAMETER_SETTINGS.items() if not v["enabled"]] or ["none"]}')
print(f'  Mgmt interventions: {[k for k,v in MANAGEMENT.items() if v] or ["none"]}')
print(f'  Species start     : {SPECIES_START_FROM!r}' + (" (all)" if not SPECIES_START_FROM else ""))


In [ ]:
# ============================================================
# CELL 4 - Load region geometry + species data
# Embedded species arrays match each region's JS file verbatim
# (extracted from var speciesRequirements). Field order is the
# 22-field positional layout used throughout the JS family.
# ============================================================

# -- Region geometry --
_field, _value = cfg['asset_filter']
_fc = ee.FeatureCollection(cfg['asset']).filter(ee.Filter.eq(_field, _value))

if cfg.get('bbox_clip'):
    _bbox = ee.Geometry.Rectangle(cfg['bbox_clip'])
    _fc   = _fc.filterBounds(_bbox)
    region_geom = _fc.geometry().intersection(_bbox, maxError=10)
else:
    region_geom = _fc.geometry()

region_area_ha = region_geom.area(maxError=1).getInfo() / 10_000
print(f'{REGION} geometry loaded | area: {region_area_ha:,.0f} ha'.format(REGION=REGION, region_area_ha=region_area_ha))

# -- Field order index (22 positional values, matches JS):
#   [0]  name
#   [1]  altMin               [2]  altOptMin    [3]  altOptMax    [4]  altMax        (m)
#   [5]  pHMin                [6]  pHOptMin     [7]  pHOptMax     [8]  pHMax
#   [9]  water_mm             [10] frostTol_C   [11] slopeMax_%
#   [12] soilPatterns         [13] drainage
#   [14] aspectOptMin         [15] aspectOptMax [16] aspectAccMin [17] aspectAccMax  (deg)
#   [18] maxTempTol_C         [19] precipWettestMin_mm
#   [20] precipSeasonalityMin_%  [21] precipSeasonalityMax_%
IDX = {
    'name': 0, 'altMin': 1, 'altOptMin': 2, 'altOptMax': 3, 'altMax': 4,
    'pHMin': 5, 'pHOptMin': 6, 'pHOptMax': 7, 'pHMax': 8,
    'water': 9, 'frostTol': 10, 'slopeMax': 11,
    'soilPatterns': 12, 'drainage': 13,
    'aspectOptMin': 14, 'aspectOptMax': 15, 'aspectAccMin': 16, 'aspectAccMax': 17,
    'maxTempTol': 18, 'precipWettestMin': 19,
    'seasonalityMin': 20, 'seasonalityMax': 21,
}

# -- Embedded species arrays (one per supported region) --
# Extracted from each region's Parameter_Species_Suitability_Tools-*-export.js
# (var speciesRequirements). Total: 128 species across 5 regions.
REGION_SPECIES = {
    'Mongolia': (
        [











          ['Abies sibirica', 100, 700, 1800, 2300, 4, 4.8, 6.2, 7, 550, -50, 55,
           'Sandy Loam|Loam|Silt Loam|Clay Loam', 'well-drained|moderately-drained|moist',
           0, 90, 0, 360,
           30, 70, 50, 110],
          ['Betula pendula', 50, 400, 1500, 2100, 4, 4.8, 6.5, 7.5, 450, -45, 60,
           'Sand|Loamy Sand|Sandy Loam|Loam|Silt Loam|Clay Loam', 'well-drained|moderately-drained|moist',
           0, 360, 0, 360,
           35, 50, 40, 130],
          ['Betula platyphylla', 200, 1100, 1900, 2400, 4.5, 5.5, 7, 7.5, 400, -50, 55,
           'Sandy Loam|Loam|Silt Loam|Clay Loam|Loamy Sand|Sandy Clay Loam', 'well-drained|moderately-drained|moist',
           0, 90, 0, 360,
           35, 50, 70, 140],
          ['Caragana arborescens', 300, 600, 2000, 2800, 5, 6, 7.5, 8.5, 300, -50, 70,
           'Sand|Loamy Sand|Sandy Loam|Loam|Silt Loam|Clay Loam|Sandy Clay Loam', 'well-drained|moderately-drained|excessively-drained',
           0, 360, 0, 360,
           40, 30, 60, 140],
          ['Caragana microphylla', 400, 800, 1800, 2500, 6, 6.5, 8.5, 9, 300, -40, 80,
           'Sand|Loamy Sand|Sandy Loam|Loam|Sandy Clay Loam', 'well-drained|excessively-drained|moderately-drained',
           90, 270, 0, 360,
           40, 40, 90, 160],
          ['Elaeagnus angustifolia', 100, 500, 1800, 2400, 6, 7, 8.5, 9.5, 250, -45, 50,
           'Sand|Loamy Sand|Sandy Loam|Loam|Silt Loam|Clay Loam|Sandy Clay Loam', 'well-drained|moderately-drained|excessively-drained|moist',
           90, 270, 0, 360,
           45, 25, 50, 140],
          ['Haloxylon ammodendron', 100, 500, 1300, 1600, 7, 7.5, 8.8, 9.5, 100, -40, 30,
           'Sand|Loamy Sand|Sandy Loam', 'excessively-drained|well-drained',
           135, 225, 0, 360,
           50, 15, 80, 160],
          ['Hippophae rhamnoides', 200, 1200, 2500, 4800, 5.5, 6.5, 8.2, 9, 400, -43, 70,
           'Sandy Loam|Loam|Loamy Sand|Silt Loam|Sandy Clay Loam|Clay Loam', 'well-drained|moderately-drained|moist',
           0, 180, 0, 360,
           40, 30, 70, 150],
          ['Larix gmelinii', 50, 300, 1200, 1800, 4, 5, 7, 7.8, 400, -70, 65,
           'Sand|Loamy Sand|Sandy Loam|Loam|Silt Loam|Clay Loam', 'well-drained|moderately-drained|moist|poorly-drained',
           0, 360, 0, 360,
           35, 50, 70, 140],
          ['Larix sibirica', 200, 1100, 2200, 2700, 4.5, 5.5, 7, 7.5, 350, -50, 60,
           'Sandy Loam|Loam|Silt Loam|Loamy Sand|Clay Loam', 'well-drained|moderately-drained|moist',
           0, 90, 0, 360,
           37, 40, 80, 150],
          ['Picea obovata', 100, 800, 2000, 2400, 4, 4.8, 6.5, 7.2, 500, -50, 60,
           'Sandy Loam|Loam|Silt Loam|Clay Loam|Silty Clay Loam', 'well-drained|moderately-drained|moist',
           0, 90, 0, 360,
           32, 60, 50, 120],
          ['Pinus sibirica', 100, 1200, 2000, 2400, 4.5, 5, 6.5, 7, 600, -60, 55,
           'Sandy Loam|Loam|Loamy Sand|Silt Loam|Clay Loam', 'well-drained|moderately-drained|moist',
           0, 90, 0, 360,
           35, 60, 40, 100],
          ['Pinus sylvestris', 100, 800, 2000, 2600, 4, 4.5, 6.5, 7.5, 500, -50, 60,
           'Sand|Loamy Sand|Sandy Loam|Loam|Sandy Clay Loam', 'well-drained|excessively-drained|moderately-drained',
           0, 180, 0, 360,
           40, 50, 30, 140],
          ['Populus euphratica', 200, 800, 1800, 4000, 7, 7.5, 8.8, 9.5, 50, -40, 25,
           'Sand|Loamy Sand|Sandy Loam|Loam|Silt Loam|Silty Clay Loam', 'moist|moderately-drained|poorly-drained|well-drained',
           0, 360, 0, 360,
           50, 10, 80, 160],
          ['Populus laurifolia', 800, 1200, 2200, 2400, 6.5, 7, 8, 8.5, 200, -45, 30,
           'Sand|Loamy Sand|Sandy Loam|Loam|Silt Loam', 'moist|moderately-drained|poorly-drained|well-drained',
           0, 360, 0, 360,
           35, 30, 90, 150],
          ['Populus sibirica', 500, 900, 1500, 2000, 6, 6.8, 8, 8.8, 300, -45, 25,
           'Sand|Loamy Sand|Sandy Loam|Loam|Silt Loam', 'well-drained|moderately-drained|moist',
           0, 360, 0, 360,
           38, 40, 60, 140],
          ['Populus tremula', 50, 400, 1600, 2200, 4.5, 5, 7, 7.8, 400, -45, 55,
           'Sandy Loam|Loam|Silt Loam|Clay Loam|Silty Clay Loam', 'well-drained|moderately-drained|moist',
           0, 180, 0, 360,
           33, 50, 40, 130],
          ['Tamarix ramosissima', 100, 500, 1500, 2300, 6.5, 7.5, 9, 9.8, 200, -38, 20,
           'Sand|Loamy Sand|Sandy Loam|Loam|Silt Loam|Sandy Clay Loam|Clay Loam|Silty Clay|Clay', 'well-drained|moderately-drained|moist|poorly-drained|excessively-drained',
           90, 270, 0, 360,
           46, 15, 50, 150],
          ['Ulmus macrocarpa', 200, 600, 1600, 2100, 5.5, 6.5, 8, 8.8, 300, -45, 60,
           'Sand|Loamy Sand|Sandy Loam|Loam|Clay Loam|Sandy Clay Loam', 'well-drained|moderately-drained|excessively-drained',
           0, 360, 0, 360,
           40, 40, 70, 140],
          ['Ulmus pumila', 300, 900, 1800, 2400, 5.5, 6.5, 8, 8.5, 250, -40, 70,
           'Sand|Loamy Sand|Sandy Loam|Loam|Silt Loam|Clay Loam|Sandy Clay Loam', 'well-drained|moderately-drained|excessively-drained',
           135, 225, 90, 270,
           42, 30, 60, 150]
        ]
    ),
    'ArunachalPradesh': (
        [











          ['Abies densa', 2800, 3000, 3700, 4000, 3.8, 4.2, 5.3, 5.8, 2200, -30, 85,
           'Loam|Silt Loam', 'well-drained|moderately-drained',
           0, 90, 0, 270,
           22, 350, 60, 110],
          ['Abies pindrow', 2200, 2600, 3300, 3700, 4.5, 5, 6.5, 7, 1800, -30, 80,
           'Loam|Silt Loam', 'well-drained|moderately-drained',
           0, 45, 0, 270,
           26, 350, 80, 130],
          ['Acer caesium', 1800, 2100, 2700, 2900, 5, 5.8, 7, 7.5, 1700, -22, 75,
           'Loam|Silt Loam|Clay Loam', 'well-drained|moderately-drained',
           0, 90, 0, 270,
           30, 340, 80, 130],
          ['Aesculus indica', 1100, 1500, 2500, 3000, 5.5, 6, 7.2, 7.8, 1700, -15, 60,
           'Loam|Silt Loam|Sandy Loam|Clay Loam', 'well-drained|moderately-drained|moist',
           0, 90, 0, 360,
           35, 350, 80, 130],
          ['Carpinus viminea', 1400, 1700, 2500, 2700, 5, 5.8, 6.8, 7.5, 1600, -20, 80,
           'Loam|Silt Loam|Clay Loam', 'well-drained|moderately-drained',
           0, 90, 0, 270,
           32, 320, 80, 130],
          ['Cedrus deodara', 1200, 1800, 2600, 3300, 5.5, 6, 7, 7.8, 1400, -25, 75,
           'Loam|Sandy Loam|Silt Loam', 'well-drained|excessively-drained',
           0, 90, 0, 360,
           32, 300, 80, 150],
          ['Cryptomeria japonica', 800, 1200, 2000, 2800, 4.5, 5, 6, 6.5, 2000, -15, 50,
           'Loam|Silt Loam|Clay Loam', 'moist|moderately-drained',
           0, 90, 0, 360,
           32, 500, 30, 70],
          ['Diospyros lotus', 700, 1000, 2000, 2400, 5.5, 6, 7.5, 8, 1100, -18, 70,
           'Loam|Sandy Loam|Silt Loam|Clay Loam', 'well-drained|excessively-drained|moderately-drained',
           0, 360, 0, 360,
           40, 230, 80, 140],
          ['Juglans regia', 900, 1200, 2200, 2500, 5.5, 6.5, 7.5, 8, 1200, -20, 60,
           'Loam|Silt Loam|Sandy Loam|Clay Loam', 'well-drained|moderately-drained',
           0, 90, 0, 270,
           38, 260, 80, 140],
          ['Larix griffithii', 2700, 3000, 3600, 4000, 4, 4.5, 6, 6.5, 2000, -30, 85,
           'Loam|Sandy Loam|Silt Loam', 'well-drained|excessively-drained',
           90, 270, 0, 360,
           22, 300, 60, 110],
          ['Magnolia campbellii', 2200, 2400, 2800, 3000, 4, 4.5, 5.8, 6.5, 2800, -15, 80,
           'Loam|Silt Loam|Clay Loam', 'well-drained|moderately-drained',
           0, 90, 0, 270,
           26, 450, 60, 110],
          ['Picea smithiana', 2100, 2400, 3100, 3500, 4.5, 5.5, 6.8, 7.2, 1500, -28, 80,
           'Loam|Sandy Loam|Silt Loam', 'well-drained|moderately-drained',
           0, 90, 0, 270,
           28, 320, 80, 130],
          ['Pinus wallichiana', 1800, 2200, 2800, 3500, 5, 5.5, 6.5, 7, 1200, -15, 60,
           'Sandy Loam|Loamy Sand|Sand', 'well-drained',
           180, 270, 0, 360,
           28, 300, 50, 100],
          ['Quercus floribunda', 1700, 2100, 2700, 2900, 5, 5.8, 7, 7.5, 1600, -18, 80,
           'Loam|Silt Loam|Clay Loam|Sandy Loam', 'well-drained|moderately-drained',
           0, 90, 0, 270,
           32, 320, 80, 130],
          ['Tsuga dumosa', 2300, 2600, 3200, 3500, 4.5, 5, 6, 6.8, 2000, -22, 80,
           'Loam|Silt Loam', 'well-drained|moderately-drained',
           0, 45, 0, 270,
           25, 400, 60, 110]
        ]
    ),
    'Rajasthan': (
        [











          ['Acacia catechu', 0, 200, 900, 1400, 6, 6.5, 7.5, 8.5, 500, -2, 50,
           '.*', '.*',
           180, 225, 180, 225,
           42, 300, 60, 100],
          ['Acacia senegal', 0, 100, 700, 1700, 5.5, 7, 8.5, 9, 280, -3, 45,
           'Sand|Loamy Sand|Sandy Loam|Loam', 'well-drained|excessively-drained',
           180, 270, 0, 360,
           50, 60, 110, 180],
          ['Aegle marmelos', 0, 0, 800, 1200, 5, 6.5, 8, 10, 800, -7, 30,
           '.*', '.*',
           180, 225, 180, 225,
           48, 250, 70, 120],
          ['Aesculus indica', 1100, 1500, 2500, 3000, 5.5, 6, 7.2, 7.8, 1700, -15, 60,
           'Loam|Silt Loam|Sandy Loam|Clay Loam', 'well-drained|moderately-drained|moist',
           0, 90, 0, 360,
           35, 350, 80, 130],
          ['Anogeissus pendula', 150, 200, 900, 1400, 6, 6.8, 8, 9, 500, -3, 70,
           'Sandy Loam|Loam|Clay Loam|Sandy Clay Loam|Silt Loam', 'well-drained|moderately-drained|excessively-drained',
           135, 270, 0, 360,
           48, 120, 90, 170],
          ['Boswellia serrata', 100, 250, 900, 1500, 6, 6.5, 7.8, 8.5, 550, -2, 75,
           'Sandy Loam|Loam|Sandy Clay Loam|Silt Loam', 'well-drained|excessively-drained',
           135, 270, 0, 360,
           47, 130, 90, 170],
          ['Capparis decidua', 0, 50, 700, 1500, 6.5, 7.5, 9, 10, 200, -5, 55,
           'Sand|Loamy Sand|Sandy Loam|Loam|Sandy Clay Loam', 'well-drained|excessively-drained|moderately-drained',
           180, 270, 0, 360,
           50, 50, 110, 180],
          ['Castanopsis hystrix', 700, 900, 1800, 2200, 4, 4.5, 5.8, 6.5, 2500, -3, 75,
           'Loam|Sandy Loam|Clay Loam', 'well-drained|moderately-drained',
           0, 360, 0, 360,
           33, 380, 50, 100],
          ['Dalbergia sissoo', 0, 100, 900, 1400, 5.5, 6.5, 7.5, 8.5, 500, -2, 30,
           'Sandy Loam|Loam|Loamy Sand', 'well-drained|moderately-drained',
           180, 270, 180, 270,
           40, 250, 60, 100],
          ['Diospyros lotus', 700, 1000, 2000, 2400, 5.5, 6, 7.5, 8, 1100, -18, 70,
           'Loam|Sandy Loam|Silt Loam|Clay Loam', 'well-drained|excessively-drained|moderately-drained',
           0, 360, 0, 360,
           40, 230, 80, 140],
          ['Emblica officinalis', 0, 200, 1200, 1800, 5, 5.5, 7, 8, 700, -2, 50,
           '.*', 'well-drained|moderately-drained',
           180, 225, 180, 225,
           40, 300, 60, 110],
          ['Garcinia xanthochymus', 0, 600, 800, 1000, 5.5, 6, 7.5, 8, 1500, -2, 40,
           '.*', 'well-drained',
           0, 360, 0, 360,
           36, 550, 30, 70],
          ['Juglans regia', 900, 1200, 2200, 2500, 5.5, 6.5, 7.5, 8, 1200, -20, 60,
           'Loam|Silt Loam|Sandy Loam|Clay Loam', 'well-drained|moderately-drained',
           0, 90, 0, 270,
           38, 260, 80, 140],
          ['Melia azedarach', 0, 200, 1200, 1800, 5, 5.5, 7, 8, 600, -5, 40,
           '.*', '.*',
           90, 180, 90, 180,
           38, 300, 60, 100],
          ['Morus indica', 0, 300, 1200, 1800, 5.5, 6, 7.5, 8.5, 800, -5, 40,
           'Loam|Sandy Loam|Silt Loam', 'well-drained|moderately-drained',
           0, 360, 0, 360,
           38, 300, 50, 100],
          ['Prosopis cineraria', 0, 100, 600, 1500, 6.5, 7.5, 8.5, 9.8, 300, -6, 40,
           'Sand|Loamy Sand|Sandy Loam|Loam|Sandy Clay Loam', 'well-drained|excessively-drained|moderately-drained',
           180, 270, 0, 360,
           50, 60, 100, 180],
          ['Salvadora oleoides', 0, 50, 500, 1200, 7, 8, 9.5, 10.5, 250, -3, 25,
           'Sand|Loamy Sand|Sandy Loam|Loam|Sandy Clay Loam|Sandy Clay|Clay', 'well-drained|moderately-drained|poorly-drained|excessively-drained',
           180, 270, 0, 360,
           50, 60, 110, 180],
          ['Tecomella undulata', 0, 100, 700, 1300, 6.5, 7.5, 8.5, 9.5, 300, -4, 40,
           'Sand|Loamy Sand|Sandy Loam|Loam', 'well-drained|excessively-drained|moderately-drained',
           180, 270, 0, 360,
           48, 70, 100, 180],
          ['Toona ciliata', 200, 500, 1200, 1500, 5.5, 6, 7, 7.5, 1500, -2, 40,
           'Loam|Silt Loam|Clay Loam', 'moderately-drained',
           0, 90, 0, 360,
           35, 450, 40, 80],
          ['Vachellia nilotica', 0, 50, 600, 1500, 5, 7, 8.5, 9.5, 400, -4, 40,
           'Sand|Loamy Sand|Sandy Loam|Loam|Clay Loam|Sandy Clay Loam|Sandy Clay|Clay', 'well-drained|moderately-drained|poorly-drained|excessively-drained',
           180, 270, 0, 360,
           50, 80, 90, 180],
          ['Vachellia tortilis', 0, 100, 800, 1900, 6, 7, 8.5, 9.3, 200, -4, 50,
           'Sand|Loamy Sand|Sandy Loam|Loam', 'well-drained|excessively-drained',
           180, 270, 0, 360,
           50, 50, 110, 180],
          ['Ziziphus mauritiana', 0, 50, 800, 1500, 5.5, 6.5, 8.5, 9.5, 350, -3, 40,
           'Sand|Loamy Sand|Sandy Loam|Loam|Sandy Clay Loam|Clay Loam', 'well-drained|moderately-drained|excessively-drained',
           180, 270, 0, 360,
           48, 80, 100, 180]
        ]
    ),
    'Sundarbans': (
        [











          ['Acacia catechu', 0, 200, 900, 1400, 6, 6.5, 7.5, 8.5, 500, -2, 50,
           '.*', '.*',
           180, 225, 180, 225,
           42, 300, 60, 100],
          ['Acacia senegal', 0, 100, 700, 1700, 5.5, 7, 8.5, 9, 280, -3, 45,
           'Sand|Loamy Sand|Sandy Loam|Loam', 'well-drained|excessively-drained',
           180, 270, 0, 360,
           50, 60, 110, 180],
          ['Aegiceras corniculatum', 0, 0, 3, 6, 6, 6.5, 8, 8.5, 1500, 5, 4,
           'Silty Clay|Clay|Silt Loam|Clay Loam|Sandy Loam', 'poorly-drained|very-poorly-drained|moist',
           0, 360, 0, 360,
           38, 250, 50, 140],
          ['Aegle marmelos', 0, 0, 800, 1200, 5, 6.5, 8, 10, 800, -7, 30,
           '.*', '.*',
           180, 225, 180, 225,
           48, 250, 70, 120],
          ['Anacardium occidentale', 0, 0, 400, 800, 4.5, 5.5, 6.5, 7.5, 1000, 5, 30,
           'Sandy Loam|Loamy Sand|Sand', 'well-drained',
           180, 270, 180, 270,
           38, 400, 40, 90],
          ['Aquilaria malaccensis', 0, 200, 800, 1200, 4.5, 5, 6, 6.5, 2000, 5, 40,
           'Loam|Silt Loam|Sandy Loam', 'well-drained|moderately-drained',
           0, 360, 0, 360,
           35, 600, 30, 70],
          ['Artocarpus chaplasha', 0, 100, 1200, 1650, 4, 5, 6.5, 7.5, 3000, 2, 40,
           'Loam|Silt Loam|Clay Loam', 'moderately-drained|moist',
           45, 90, 0, 360,
           32, 600, 30, 70],
          ['Artocarpus heterophyllus', 0, 100, 800, 1200, 5.5, 6, 7, 7.5, 1500, 2, 30,
           'Loam|Silt Loam|Clay Loam', 'moderately-drained',
           90, 135, 90, 135,
           36, 550, 30, 70],
          ['Avicennia marina', 0, 0, 3, 8, 6, 7, 8.2, 8.8, 1500, 5, 3,
           'Silty Clay|Clay|Silt Loam|Sandy Loam|Sand', 'poorly-drained|very-poorly-drained|moist',
           0, 360, 0, 360,
           40, 200, 40, 140],
          ['Avicennia officinalis', 0, 0, 3, 7, 6, 6.8, 8, 8.5, 1700, 8, 3,
           'Silty Clay|Clay|Silt Loam', 'poorly-drained|very-poorly-drained|moist',
           0, 360, 0, 360,
           38, 300, 50, 130],
          ['Azadirachta indica', 0, 100, 700, 1000, 5.5, 6.5, 7.5, 8.5, 400, 4, 30,
           '.*', '.*',
           180, 225, 180, 225,
           42, 250, 70, 120],
          ['Bambusa balcooa', 0, 100, 600, 900, 5, 5.5, 7, 7.5, 1500, 2, 40,
           'Loam|Clay Loam|Silt Loam', 'moderately-drained|moist',
           90, 180, 90, 180,
           36, 550, 30, 70],
          ['Bambusa bambos', 0, 200, 800, 1200, 6, 6.5, 7.5, 8.5, 800, 0, 50,
           '.*', '.*',
           180, 270, 0, 360,
           40, 300, 60, 110],
          ['Bambusa tulda', 0, 100, 800, 1200, 5, 5.5, 6.5, 7.5, 1500, 0, 50,
           'Loam|Silt Loam|Clay Loam', 'moist',
           45, 90, 45, 90,
           38, 550, 30, 70],
          ['Barringtonia acutangula', 0, 100, 600, 1000, 5, 5.5, 7, 8, 1500, 5, 30,
           'Loam|Silt Loam|Clay Loam', 'moist|poorly-drained',
           0, 360, 0, 360,
           36, 600, 30, 70],
          ['Bombax ceiba', 0, 100, 800, 1200, 5.5, 6, 7, 7.5, 1500, 0, 30,
           'Loam|Clay Loam|Silt Loam', 'moist',
           90, 180, 90, 180,
           40, 400, 70, 110],
          ['Bruguiera gymnorrhiza', 0, 1, 4, 8, 6, 6.8, 8, 8.5, 1700, 8, 4,
           'Silty Clay|Clay|Silt Loam|Clay Loam', 'poorly-drained|very-poorly-drained|moist',
           0, 360, 0, 360,
           38, 280, 50, 130],
          ['Butea monosperma', 0, 200, 900, 1200, 5.5, 6, 7.5, 8.5, 750, 0, 40,
           '.*', '.*',
           180, 225, 180, 225,
           40, 300, 70, 110],
          ['Capparis decidua', 0, 50, 700, 1500, 6.5, 7.5, 9, 10, 200, -5, 55,
           'Sand|Loamy Sand|Sandy Loam|Loam|Sandy Clay Loam', 'well-drained|excessively-drained|moderately-drained',
           180, 270, 0, 360,
           50, 50, 110, 180],
          ['Ceriops decandra', 0, 1, 4, 7, 6, 6.8, 8, 8.5, 1700, 8, 4,
           'Silty Clay|Clay|Silt Loam|Clay Loam', 'poorly-drained|very-poorly-drained|moist',
           0, 360, 0, 360,
           38, 280, 50, 130],
          ['Chukrasia tabularis', 0, 300, 1000, 1500, 5.5, 6, 7, 7.5, 1500, 2, 40,
           'Loam|Silt Loam|Clay Loam', 'well-drained|moderately-drained',
           45, 135, 0, 360,
           36, 450, 40, 80],
          ['Citrus reticulata', 0, 200, 800, 1500, 5.5, 6, 6.5, 7.5, 1200, 0, 30,
           'Loam|Sandy Loam|Silt Loam', 'well-drained',
           90, 135, 90, 135,
           36, 350, 40, 80],
          ['Citrus species', 0, 100, 800, 1500, 5.5, 6, 6.5, 7.5, 1200, 2, 30,
           'Loam|Sandy Loam|Silt Loam', 'well-drained|moderately-drained',
           90, 135, 90, 135,
           36, 350, 40, 80],
          ['Dalbergia sissoo', 0, 100, 900, 1400, 5.5, 6.5, 7.5, 8.5, 500, -2, 30,
           'Sandy Loam|Loam|Loamy Sand', 'well-drained|moderately-drained',
           180, 270, 180, 270,
           40, 250, 60, 100],
          ['Dendrocalamus strictus', 0, 200, 800, 1200, 5.5, 6, 7, 8, 750, 0, 50,
           '.*', '.*',
           180, 225, 180, 225,
           40, 350, 60, 100],
          ['Dillenia indica', 0, 200, 800, 1200, 5, 5.5, 6.5, 7, 1800, 2, 40,
           'Loam|Silt Loam|Clay Loam', 'moist|moderately-drained',
           45, 135, 0, 360,
           36, 550, 30, 70],
          ['Duabanga grandiflora', 0, 900, 1200, 1500, 4.5, 5, 6, 6.5, 2000, 0, 50,
           'Loam|Silt Loam|Clay Loam', 'moist|poorly-drained',
           0, 45, 0, 360,
           33, 500, 30, 70],
          ['Elaeocarpus serratus', 0, 200, 1000, 1500, 5.5, 6, 7, 7.5, 2000, 0, 40,
           'Loam|Sandy Loam|Silt Loam', 'moist|moderately-drained',
           45, 90, 45, 90,
           35, 500, 30, 70],
          ['Emblica officinalis', 0, 200, 1200, 1800, 5, 5.5, 7, 8, 700, -2, 50,
           '.*', 'well-drained|moderately-drained',
           180, 225, 180, 225,
           40, 300, 60, 110],
          ['Excoecaria agallocha', 0, 1, 5, 10, 6, 6.5, 8, 8.5, 1700, 7, 5,
           'Silty Clay|Clay|Silt Loam|Sandy Clay Loam', 'poorly-drained|moist|well-drained',
           0, 360, 0, 360,
           38, 280, 50, 130],
          ['Garcinia xanthochymus', 0, 600, 800, 1000, 5.5, 6, 7.5, 8, 1500, -2, 40,
           '.*', 'well-drained',
           0, 360, 0, 360,
           36, 550, 30, 70],
          ['Gmelina arborea', 0, 100, 700, 1000, 5, 5.5, 7, 8, 1000, 2, 30,
           'Loam|Silt Loam|Clay Loam', 'moderately-drained',
           90, 180, 90, 180,
           38, 400, 60, 100],
          ['Heritiera fomes', 0, 0, 3, 8, 6, 6.5, 7.8, 8.2, 1800, 8, 3,
           'Silty Clay|Clay|Silt Loam|Clay Loam', 'poorly-drained|very-poorly-drained|moist',
           0, 360, 0, 360,
           36, 300, 60, 130],
          ['Hevea brasiliensis', 0, 100, 400, 700, 4, 4.5, 5.5, 6.5, 2000, 10, 20,
           'Loam|Clay Loam|Sandy Clay Loam', 'moderately-drained',
           45, 90, 0, 315,
           35, 700, 20, 50],
          ['Lagerstroemia parviflora', 0, 200, 800, 1200, 5.5, 6, 7, 7.5, 1000, 0, 40,
           '.*', 'well-drained|moderately-drained',
           180, 270, 0, 360,
           38, 350, 60, 100],
          ['Lagerstroemia speciosa', 0, 200, 800, 1200, 5.5, 6, 7, 7.5, 1200, 0, 40,
           '.*', 'well-drained|moderately-drained',
           180, 270, 0, 360,
           38, 400, 60, 100],
          ['Litchi chinensis', 0, 100, 500, 800, 5.5, 6, 6.5, 7.5, 1500, 2, 20,
           'Loam|Silt Loam|Clay Loam', 'moderately-drained',
           0, 90, 0, 90,
           36, 450, 40, 80],
          ['Mangifera indica', 0, 100, 600, 1000, 5.5, 6, 7, 7.5, 1000, 2, 20,
           'Loam|Silt Loam|Clay Loam', 'moderately-drained',
           135, 180, 135, 180,
           38, 450, 40, 80],
          ['Melia azedarach', 0, 200, 1200, 1800, 5, 5.5, 7, 8, 600, -5, 40,
           '.*', '.*',
           90, 180, 90, 180,
           38, 300, 60, 100],
          ['Melocanna baccifera', 0, 200, 1000, 1800, 5, 5.5, 6.5, 7, 1800, 0, 50,
           'Loam|Silt Loam|Clay Loam', 'moist|moderately-drained',
           45, 90, 45, 90,
           34, 500, 30, 70],
          ['Morus indica', 0, 300, 1200, 1800, 5.5, 6, 7.5, 8.5, 800, -5, 40,
           'Loam|Sandy Loam|Silt Loam', 'well-drained|moderately-drained',
           0, 360, 0, 360,
           38, 300, 50, 100],
          ['Neolamarckia cadamba', 0, 100, 800, 1200, 5, 5.5, 6.5, 7.5, 1500, 2, 30,
           'Loam|Clay Loam|Silt Loam', 'moist',
           0, 90, 0, 360,
           38, 550, 30, 70],
          ['Parkia timoriana', 0, 0, 600, 1300, 5, 5.5, 6.5, 7, 2000, 0, 50,
           'Loam|Silt Loam|Clay Loam', 'moderately-drained|moist',
           0, 360, 0, 360,
           35, 500, 30, 70],
          ['Phoenix paludosa', 0, 1, 5, 10, 6, 6.8, 8, 8.5, 1700, 7, 5,
           'Silty Clay|Clay|Silt Loam|Clay Loam|Sandy Loam', 'poorly-drained|moist|well-drained',
           0, 360, 0, 360,
           40, 280, 50, 130],
          ['Prosopis cineraria', 0, 100, 600, 1500, 6.5, 7.5, 8.5, 9.8, 300, -6, 40,
           'Sand|Loamy Sand|Sandy Loam|Loam|Sandy Clay Loam', 'well-drained|excessively-drained|moderately-drained',
           180, 270, 0, 360,
           50, 60, 100, 180],
          ['Psidium guajava', 0, 100, 800, 1500, 5, 5.5, 7, 8, 1000, 2, 30,
           '.*', '.*',
           90, 180, 90, 180,
           38, 400, 40, 90],
          ['Rhizophora apiculata', 0, 0, 2, 5, 6, 6.5, 8, 8.5, 2000, 10, 3,
           'Silty Clay|Clay|Silt Loam', 'poorly-drained|very-poorly-drained',
           0, 360, 0, 360,
           38, 350, 60, 130],
          ['Rhizophora mucronata', 0, 0, 2, 5, 6, 6.5, 8, 8.5, 1800, 10, 3,
           'Silty Clay|Clay|Silt Loam', 'poorly-drained|very-poorly-drained',
           0, 360, 0, 360,
           38, 300, 60, 130],
          ['Salvadora oleoides', 0, 50, 500, 1200, 7, 8, 9.5, 10.5, 250, -3, 25,
           'Sand|Loamy Sand|Sandy Loam|Loam|Sandy Clay Loam|Sandy Clay|Clay', 'well-drained|moderately-drained|poorly-drained|excessively-drained',
           180, 270, 0, 360,
           50, 60, 110, 180],
          ['Shorea robusta', 0, 200, 900, 1200, 5, 5.5, 6.5, 7.5, 1200, 0, 40,
           'Loam|Clay Loam|Silt Loam', 'moderately-drained|moist',
           90, 135, 90, 135,
           38, 400, 60, 100],
          ['Sonneratia apetala', 0, 0, 2, 5, 6, 6.5, 7.8, 8.2, 1800, 8, 3,
           'Silty Clay|Clay|Silt Loam|Sandy Loam', 'poorly-drained|very-poorly-drained|moist',
           0, 360, 0, 360,
           38, 300, 60, 130],
          ['Swietenia mahagoni', 0, 100, 600, 1000, 5.5, 6, 7, 7.5, 1500, 5, 30,
           'Loam|Silt Loam|Clay Loam', 'well-drained|moderately-drained',
           0, 360, 0, 360,
           38, 500, 40, 80],
          ['Syzygium cumini', 0, 200, 900, 1400, 5, 5.5, 7, 7.5, 1500, 0, 30,
           'Loam|Silt Loam|Clay Loam', 'moderately-drained|moist',
           45, 90, 45, 90,
           38, 500, 40, 80],
          ['Tamarindus indica', 0, 100, 500, 900, 5.5, 6.5, 7.5, 8.5, 500, 4, 20,
           'Loam|Silt Loam|Clay Loam', 'moderately-drained',
           180, 225, 180, 225,
           42, 250, 70, 120],
          ['Tecomella undulata', 0, 100, 700, 1300, 6.5, 7.5, 8.5, 9.5, 300, -4, 40,
           'Sand|Loamy Sand|Sandy Loam|Loam', 'well-drained|excessively-drained|moderately-drained',
           180, 270, 0, 360,
           48, 70, 100, 180],
          ['Tectona grandis', 0, 200, 900, 1200, 5.5, 6, 7, 8, 1200, 2, 30,
           'Loam|Clay Loam|Silty Clay Loam', 'moderately-drained',
           180, 225, 180, 225,
           40, 350, 70, 110],
          ['Terminalia arjuna', 0, 100, 700, 1000, 6, 6.5, 7.5, 8.5, 1000, 0, 20,
           'Loam|Clay Loam|Silt Loam', 'moist',
           0, 90, 0, 360,
           40, 600, 50, 90],
          ['Terminalia bellirica', 0, 200, 900, 1200, 5, 5.5, 7, 7.5, 1500, 0, 40,
           '.*', 'moist|poorly-drained',
           90, 135, 90, 135,
           38, 400, 60, 100],
          ['Terminalia chebula', 0, 200, 900, 1500, 5.5, 6, 7.5, 8, 1000, 0, 40,
           '.*', 'well-drained|moderately-drained',
           90, 180, 90, 180,
           40, 350, 60, 100],
          ['Theobroma cacao', 0, 200, 600, 1000, 5, 5.5, 6.5, 7, 1800, 10, 30,
           'Loam|Silt Loam|Clay Loam', 'well-drained|moist',
           0, 360, 0, 360,
           32, 600, 20, 50],
          ['Vachellia nilotica', 0, 50, 600, 1500, 5, 7, 8.5, 9.5, 400, -4, 40,
           'Sand|Loamy Sand|Sandy Loam|Loam|Clay Loam|Sandy Clay Loam|Sandy Clay|Clay', 'well-drained|moderately-drained|poorly-drained|excessively-drained',
           180, 270, 0, 360,
           50, 80, 90, 180],
          ['Vachellia tortilis', 0, 100, 800, 1900, 6, 7, 8.5, 9.3, 200, -4, 50,
           'Sand|Loamy Sand|Sandy Loam|Loam', 'well-drained|excessively-drained',
           180, 270, 0, 360,
           50, 50, 110, 180],
          ['Xylocarpus mekongensis', 0, 1, 4, 8, 6, 6.8, 7.8, 8.2, 1800, 9, 3,
           'Silty Clay|Clay|Silt Loam|Clay Loam', 'poorly-drained|very-poorly-drained|moist',
           0, 360, 0, 360,
           36, 300, 60, 130],
          ['Ziziphus mauritiana', 0, 50, 800, 1500, 5.5, 6.5, 8.5, 9.5, 350, -3, 40,
           'Sand|Loamy Sand|Sandy Loam|Loam|Sandy Clay Loam|Clay Loam', 'well-drained|moderately-drained|excessively-drained',
           180, 270, 0, 360,
           48, 80, 100, 180]
        ]
    ),
    'Uttarakhand': (
        [











          ['Abies pindrow', 2200, 2600, 3300, 3700, 4.5, 5, 6.5, 7, 1800, -30, 80,
           'Loam|Silt Loam', 'well-drained|moderately-drained',
           0, 45, 0, 270,
           26, 350, 80, 130],
          ['Acer caesium', 1800, 2100, 2700, 2900, 5, 5.8, 7, 7.5, 1700, -22, 75,
           'Loam|Silt Loam|Clay Loam', 'well-drained|moderately-drained',
           0, 90, 0, 270,
           30, 340, 80, 130],
          ['Carpinus viminea', 1400, 1700, 2500, 2700, 5, 5.8, 6.8, 7.5, 1600, -20, 80,
           'Loam|Silt Loam|Clay Loam', 'well-drained|moderately-drained',
           0, 90, 0, 270,
           32, 320, 80, 130],
          ['Cedrus deodara', 1200, 1800, 2600, 3300, 5.5, 6, 7, 7.8, 1400, -25, 75,
           'Loam|Sandy Loam|Silt Loam', 'well-drained|excessively-drained',
           0, 90, 0, 360,
           32, 300, 80, 150],
          ['Juglans regia', 900, 1200, 2200, 2500, 5.5, 6.5, 7.5, 8, 1200, -20, 60,
           'Loam|Silt Loam|Sandy Loam|Clay Loam', 'well-drained|moderately-drained',
           0, 90, 0, 270,
           38, 260, 80, 140],
          ['Picea smithiana', 2100, 2400, 3100, 3500, 4.5, 5.5, 6.8, 7.2, 1500, -28, 80,
           'Loam|Sandy Loam|Silt Loam', 'well-drained|moderately-drained',
           0, 90, 0, 270,
           28, 320, 80, 130],
          ['Tsuga dumosa', 2300, 2600, 3200, 3500, 4.5, 5, 6, 6.8, 2000, -22, 80,
           'Loam|Silt Loam', 'well-drained|moderately-drained',
           0, 45, 0, 270,
           25, 400, 60, 110]
        ]
    ),
}

SPECIES = REGION_SPECIES[REGION]

print(f'\n{len(SPECIES)} species available for {REGION}:'.format(len=len, SPECIES=SPECIES, REGION=REGION))
for s in SPECIES:
    print(f'    - {s[IDX["name"]]}'.format(s=s, IDX=IDX))


In [ ]:
# ============================================================
# CELL 5 - Build input data layers + scope mask + texture class
#
# All asset paths and unit conversions match the JS app where the
# JS paths were valid. Two deliberate upgrades over the JS:
#
#  1. Soil pH source: OpenLandMap (b10 = 10 cm depth) instead of
#     the SoilGrids ISRIC asset, which has a less stable GEE path.
#
#  2. Soil texture: real USDA-TT 12-class lookup computed from
#     SoilGrids sand/clay percentages via the USDA texture
#     triangle. The JS soil match was effectively a no-op
#     (returned 2 everywhere for any species with a non-empty
#     texture list); this notebook actually filters pixels by
#     species' allowed textures.
# ============================================================

import math

WC_SCALE  = 250   # WorldClim/CHIRPS reproject scale
SG_SCALE  = 250   # SoilGrids/OpenLandMap reproject scale

# -- DEM / slope / aspect (NASADEM @ 30 m) --
nasadem    = ee.Image('USGS/NASADEM_HGT/001').select('elevation')
slope_deg  = ee.Terrain.slope(nasadem)
slope_pct  = slope_deg.multiply(math.pi / 180).tan().multiply(100)
aspect_deg = ee.Terrain.aspect(nasadem)

# -- CHIRPS annual rainfall (mm/yr, 2014-2023 mean) --
chirps_collection = (
    ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
      .filter(ee.Filter.date('2014-01-01', '2023-12-31'))
)
rainfall_chirps = (chirps_collection.sum().divide(10)
                   .resample('bilinear').reproject(crs='EPSG:4326', scale=WC_SCALE))

# -- WorldClim BIO bands (V1) --
worldclim = ee.Image('WORLDCLIM/V1/BIO')
def _wc(band, factor=1.0):
    img = worldclim.select(band)
    if factor != 1.0:
        img = img.divide(factor)
    return img.resample('bilinear').reproject(crs='EPSG:4326', scale=WC_SCALE)

min_temp           = _wc('bio06', 10.0)   # deg C (BIO6 stored as deg C x 10)
max_temp           = _wc('bio05', 10.0)   # deg C
rainfall_worldclim = _wc('bio12')          # mm/yr
precip_wettest     = _wc('bio13')          # mm (wettest month)
precip_seasonality = _wc('bio15')          # CV %

# -- Projected (downscaled) precip - matches JS "Projected WorldClim 250m" --
ratio = rainfall_chirps.divide(rainfall_worldclim.add(1e-6))
rainfall_projected       = rainfall_worldclim.multiply(ratio)
precip_wettest_projected = precip_wettest.multiply(ratio)

# -- Choose precipitation source per config --
if PRECIP_SOURCE == 'CHIRPS':
    rainfall_source, precip_wettest_source = rainfall_chirps, precip_wettest
elif PRECIP_SOURCE == 'WORLDCLIM':
    rainfall_source, precip_wettest_source = rainfall_worldclim, precip_wettest
elif PRECIP_SOURCE == 'PROJECTED':
    rainfall_source, precip_wettest_source = rainfall_projected, precip_wettest_projected
else:
    raise ValueError(f'PRECIP_SOURCE must be CHIRPS|WORLDCLIM|PROJECTED, got {PRECIP_SOURCE!r}')

# -- Soil pH: OpenLandMap H2O pH at 10 cm depth (values stored as pH x 10) --
soil_ph = (ee.Image('OpenLandMap/SOL/SOL_PH-H2O_USDA-4C1A2A_M/v02')
           .select('b10').divide(10)
           .resample('bilinear').reproject(crs='EPSG:4326', scale=SG_SCALE))

# -- Soil texture: USDA-TT 12-class from SoilGrids sand/clay percentages --
soil_sand = (ee.Image('projects/soilgrids-isric/sand_mean')
             .select('sand_0-5cm_mean').divide(10)
             .resample('bilinear').reproject(crs='EPSG:4326', scale=SG_SCALE))
soil_clay = (ee.Image('projects/soilgrids-isric/clay_mean')
             .select('clay_0-5cm_mean').divide(10)
             .resample('bilinear').reproject(crs='EPSG:4326', scale=SG_SCALE))
# Clamp to [0, 100] - bilinear resampling between adjacent SoilGrids pixels
# whose sand+clay sums differ can produce values just outside [0, 100],
# which would let pixels slip through both the Sand rule (silt+1.5*clay<15)
# and the Silt rule (silt>=80) without landing in any class.
soil_silt = ee.Image(100).subtract(soil_sand).subtract(soil_clay).clamp(0, 100)

TEXTURE_CODES = {
    'Sand':            1,  'Loamy Sand':      2,  'Sandy Loam':      3,
    'Sandy Clay Loam': 4,  'Sandy Clay':      5,  'Loam':            6,
    'Silt Loam':       7,  'Silt':            8,  'Silty Clay Loam': 9,
    'Silty Clay':     10,  'Clay Loam':      11,  'Clay':           12,
}

def _compute_texture_class(sand, clay, silt):
    """USDA-TT 12-class soil texture from sand/clay/silt percentages."""
    cls = ee.Image(0).byte()
    def assign(mask, value):
        return cls.where(cls.eq(0).And(mask), value)

    cls = assign(sand.gte(85).And(silt.add(clay.multiply(1.5)).lt(15)), 1)
    cls = assign((sand.gte(70).And(sand.lt(90))
                  .And(silt.add(clay.multiply(2)).gte(15))
                  .And(silt.add(clay.multiply(2)).lt(30))), 2)
    cls = assign(clay.gte(35).And(sand.gte(45)), 5)
    cls = assign(clay.gte(20).And(clay.lt(35)).And(silt.lt(28)).And(sand.gt(45)), 4)
    cls = assign(clay.lt(20).And(sand.gte(43)), 3)
    cls = assign(silt.gte(80).And(clay.lt(12)), 8)
    cls = assign(silt.gte(50).And(clay.lt(27)), 7)
    cls = assign(clay.gte(40).And(silt.gte(40)), 10)
    cls = assign(clay.gte(27).And(clay.lt(40)).And(sand.lte(20)), 9)
    cls = assign(clay.gte(40).And(sand.lt(45)).And(silt.lt(40)), 12)
    cls = assign(clay.gte(27).And(clay.lt(40)).And(sand.gt(20)).And(sand.lte(45)), 11)
    cls = assign(clay.gte(7).And(clay.lt(27))
                 .And(silt.gte(28)).And(silt.lt(50))
                 .And(sand.gte(23)).And(sand.lte(52)), 6)
    return cls

texture_class = _compute_texture_class(soil_sand, soil_clay, soil_silt)

# -- ESA WorldCover (scope mask) --
esa_world_cover = ee.ImageCollection('ESA/WorldCover/v200').first()

def _esa_in(classes):
    img = esa_world_cover.eq(classes[0])
    for c in classes[1:]:
        img = img.Or(esa_world_cover.eq(c))
    return img

scope_landcover = _esa_in(SCOPE_INCLUDE_CLASSES)
scope_slope     = slope_pct.lte(SCOPE_MAX_SLOPE_PCT)
scope_precip    = (rainfall_source.gte(SCOPE_MIN_PRECIP_MM)
                   if SCOPE_MIN_PRECIP_MM > 0 else ee.Image(1))
scope_mask      = scope_landcover.And(scope_slope).And(scope_precip)

# Resolution audit (each input -> EXPORT_SCALE_M m export)
_RES = [
    ('NASADEM elev/slope/aspect', '30 m',    'true 30 m'),
    ('ESA WorldCover',            '10 m',    'aggregated to ' + str(EXPORT_SCALE_M) + ' m'),
    ('SoilGrids sand%, clay%',    '250 m',   'bilinear -> ' + str(EXPORT_SCALE_M) + ' m'),
    ('OpenLandMap pH (10 cm)',    '250 m',   'bilinear -> ' + str(EXPORT_SCALE_M) + ' m'),
    ('WorldClim BIO',             '~927 m',  'bilinear -> ' + str(EXPORT_SCALE_M) + ' m'),
    ('CHIRPS daily',              '~5500 m', 'bilinear -> ' + str(EXPORT_SCALE_M) + ' m'),
]
print('Input data layers built')
print(f'  Region               : {REGION}')
print(f'  Precipitation source : {PRECIP_SOURCE}')
print(f'  Soil pH source       : OpenLandMap H2O pH (10 cm)')
print(f'  Soil texture         : USDA-TT triangle from SoilGrids sand/clay')
print(f'  ESA classes in scope : {SCOPE_INCLUDE_CLASSES}')
print(f'  Slope cap            : <= {SCOPE_MAX_SLOPE_PCT}%')
print(f'  Precip floor         : >= {SCOPE_MIN_PRECIP_MM} mm/yr'
      + (' (disabled)' if SCOPE_MIN_PRECIP_MM == 0 else ''))
print('')
print(f'Resolution audit (each input -> {EXPORT_SCALE_M} m export):')
for name, native, effective in _RES:
    print(f'  {name:<28}  native={native:<10}  {effective}')


In [ ]:
# ============================================================
# CELL 6 - Scoring functions
# Verbatim port of calculateSpeciesSuitability and
# calculateLimitingFactors from the JS app (lines 1991+ and
# 2811+ respectively), with one improvement: soil_match_score()
# performs a real per-pixel USDA-TT class lookup instead of
# the JS no-op behaviour. See Cell 5 for the texture class
# computation.
#
# Limiting-factor code legend (matches JS):
#   1  Climate - cold/frost                  9  Climate + Water
#   2  Climate - heat                       10  Climate + Terrain
#   3  Water   - rainfall too low           11  Water  + Terrain
#   4  Water   - seasonality mismatch       12  Soil   + Terrain
#   5  Terrain - slope too steep            13  Climate + Soil
#   6  Terrain - aspect unfavourable        14  Multi-factor (3+ themes)
#   7  Soil    - texture/drainage           15  Management-limited (marginal
#   8  Soil    - pH out of range                 but unmanageable)
# ============================================================

def zone_aspect_score(aspect_img, opt_min, opt_max, acc_min, acc_max):
    """Aspect -> 0/1/2 within optional/acceptable bands."""
    if opt_min == 0 and opt_max == 360:
        return ee.Image(2)
    if opt_min == 0 and opt_max == 0:
        return ee.Image(0)
    in_opt = aspect_img.gte(opt_min).And(aspect_img.lte(opt_max))
    in_acc = aspect_img.gte(acc_min).And(aspect_img.lte(acc_max))
    return ee.Image(0).where(in_acc, 1).where(in_opt, 2)


def soil_match_score(species):
    """Real USDA-TT texture-class match per pixel.

    Returns 0 (texture class not in species' allowed list) or 2 (matches).
    Binary, like the JS implementation (the JS never produced score 1 from
    its soil branch - see lines 3055+, max(textureMatch, drainageMatch, 2)
    floored to 2 unless both were 0).

    If a species has no recognised texture entries, returns 2 everywhere
    (no constraint).
    """
    species_textures = [t for t in species[IDX['soilPatterns']].split('|')
                        if t in TEXTURE_CODES]
    if not species_textures:
        return ee.Image(2)

    allowed_codes = [TEXTURE_CODES[t] for t in species_textures]
    in_allowed = texture_class.eq(allowed_codes[0])
    for code in allowed_codes[1:]:
        in_allowed = in_allowed.Or(texture_class.eq(code))

    return ee.Image(0).where(in_allowed, 2)


def calculate_species_suitability(species):
    """Returns (overall_suitability_image, per_parameter_scores_dict)."""
    name           = species[IDX['name']]
    alt_min        = species[IDX['altMin']]
    alt_opt_min    = species[IDX['altOptMin']]
    alt_opt_max    = species[IDX['altOptMax']]
    alt_max        = species[IDX['altMax']]
    ph_min         = species[IDX['pHMin']]
    ph_opt_min     = species[IDX['pHOptMin']]
    ph_opt_max     = species[IDX['pHOptMax']]
    ph_max         = species[IDX['pHMax']]
    water_req      = species[IDX['water']]
    frost_tol      = species[IDX['frostTol']]
    slope_max      = species[IDX['slopeMax']]
    asp_opt_min    = species[IDX['aspectOptMin']]
    asp_opt_max    = species[IDX['aspectOptMax']]
    asp_acc_min    = species[IDX['aspectAccMin']]
    asp_acc_max    = species[IDX['aspectAccMax']]
    max_temp_tol   = species[IDX['maxTempTol']]
    pw_min         = species[IDX['precipWettestMin']]
    seas_min       = species[IDX['seasonalityMin']]
    seas_max       = species[IDX['seasonalityMax']]

    # Altitude - hard
    alt_opt  = nasadem.gte(alt_opt_min).And(nasadem.lte(alt_opt_max))
    alt_marg = nasadem.gte(alt_min).And(nasadem.lte(alt_max)).And(alt_opt.Not())
    alt_score = ee.Image(0).where(alt_marg, 1).where(alt_opt, 2)

    # Frost - hard
    frost_marg = min_temp.lt(frost_tol).And(min_temp.gte(frost_tol - 3))
    frost_opt  = min_temp.gte(frost_tol)
    frost_score = ee.Image(0).where(frost_marg, 1).where(frost_opt, 2)

    # Heat - hard
    heat_marg = max_temp.gt(max_temp_tol).And(max_temp.lte(max_temp_tol + 3))
    heat_opt  = max_temp.lte(max_temp_tol)
    heat_score = ee.Image(0).where(heat_marg, 1).where(heat_opt, 2)

    # pH - soft (4-zone: optimal / in-range / amendable / far-out)
    ph_opt  = soil_ph.gte(ph_opt_min).And(soil_ph.lte(ph_opt_max))
    ph_in   = soil_ph.gte(ph_min).And(soil_ph.lte(ph_max))
    ph_out  = soil_ph.lt(ph_min).Or(soil_ph.gt(ph_max))
    ph_far  = soil_ph.lt(ph_min - 0.5).Or(soil_ph.gt(ph_max + 0.5))
    ph_score = (ee.Image(0)
                .where(ph_out.And(ph_far.Not()), 1)
                .where(ph_in.And(ph_opt.Not()), 1)
                .where(ph_opt, 2))

    # Rainfall - soft (60% of req = marginal floor)
    rain_marg = rainfall_source.lt(water_req).And(rainfall_source.gte(water_req * 0.6))
    rain_opt  = rainfall_source.gte(water_req)
    rain_score = ee.Image(0).where(rain_marg, 1).where(rain_opt, 2)

    # Slope - soft (130% of max = marginal ceiling)
    slope_marg = slope_pct.gt(slope_max).And(slope_pct.lte(slope_max * 1.3))
    slope_opt  = slope_pct.lte(slope_max)
    slope_score = ee.Image(0).where(slope_marg, 1).where(slope_opt, 2)

    # Aspect - soft
    aspect_score = zone_aspect_score(aspect_deg, asp_opt_min, asp_opt_max, asp_acc_min, asp_acc_max)

    # Soil - soft (real USDA-TT class match — see soil_match_score docstring)
    soil_score = soil_match_score(species)

    # Wettest month - soft (70% of min = marginal floor)
    pw_marg = precip_wettest_source.lt(pw_min).And(precip_wettest_source.gte(pw_min * 0.7))
    pw_opt  = precip_wettest_source.gte(pw_min)
    pw_score = ee.Image(0).where(pw_marg, 1).where(pw_opt, 2)

    # Seasonality - soft (+-20 CV % tolerance)
    seas_opt  = precip_seasonality.gte(seas_min).And(precip_seasonality.lte(seas_max))
    seas_tol  = 20
    seas_marg = (precip_seasonality.gte(seas_min - seas_tol)
                  .And(precip_seasonality.lte(seas_max + seas_tol))
                  .And(seas_opt.Not()))
    seas_score = ee.Image(0).where(seas_marg, 1).where(seas_opt, 2)

    # -- Combine with hard/soft + management upgrades --
    PARAM = [
        ('altitude',          alt_score,    False),
        ('frost',             frost_score,  MANAGEMENT['microclimate']),
        ('maxTemp',           heat_score,   MANAGEMENT['microclimate']),
        ('pH',                ph_score,     MANAGEMENT['soilAmendment']),
        ('rainfall',          rain_score,   MANAGEMENT['irrigation']),
        ('slope',             slope_score,  MANAGEMENT['terracing']),
        ('aspect',            aspect_score, MANAGEMENT['microclimate']),
        ('soil',              soil_score,   MANAGEMENT['drainage']),
        ('precipWettest',     pw_score,     MANAGEMENT['drainage'] or MANAGEMENT['irrigation']),
        ('precipSeasonality', seas_score,   False),
    ]

    used_scores         = []
    hard_fails          = ee.Image(0)
    soft_fails          = ee.Image(0)
    hard_marginals      = ee.Image(0)
    soft_marginals      = ee.Image(0)
    unmanaged_marginals = ee.Image(0)

    for pname, sc, manageable in PARAM:
        cfg = PARAMETER_SETTINGS[pname]
        if not cfg['enabled']:
            continue
        fail = sc.eq(0)
        marg = sc.eq(1)
        effective = sc.where(marg, 2) if manageable else sc
        used_scores.append(effective)
        if cfg['hard']:
            hard_fails     = hard_fails.add(fail)
            hard_marginals = hard_marginals.add(marg)
        else:
            soft_fails     = soft_fails.add(fail)
            soft_marginals = soft_marginals.add(marg)
            if not manageable:
                unmanaged_marginals = unmanaged_marginals.add(marg)

    if not used_scores:
        overall = ee.Image(2)
    else:
        optimal_count = ee.Image(0)
        for s in used_scores:
            optimal_count = optimal_count.add(s.eq(2))

        thresholds = ECOLOGY_THRESHOLDS[ECOLOGY_TYPE]
        opt_threshold = max(1, math.ceil(len(used_scores) * thresholds['optimalThreshold']))
        max_unmgd     = thresholds['maxUnmanagedMarginals']

        overall = ee.Image(1)                                 # start marginal
        overall = overall.where(hard_fails.gt(0), 0)          # any hard fail -> unsuitable
        overall = overall.where(soft_fails.gt(1), 0)          # 2+ soft fails -> unsuitable

        if CAUTIOUS_MODE:
            up = (optimal_count.gte(opt_threshold)
                  .And(hard_marginals.eq(0))
                  .And(unmanaged_marginals.lte(max_unmgd))
                  .And(overall.neq(0)))
        else:
            up = (optimal_count.gte(opt_threshold)
                  .And(unmanaged_marginals.lte(max_unmgd))
                  .And(overall.neq(0)))
        overall = overall.where(up, 2)

        overall = overall.where(unmanaged_marginals.gt(max_unmgd).And(overall.neq(0)), 1)

        if CAUTIOUS_MODE:
            overall = overall.where(hard_marginals.gt(0).And(overall.neq(0)), 1)

    scores_dict = {
        'alt':           alt_score,    'frost':         frost_score,
        'heat':          heat_score,   'ph':            ph_score,
        'rain':          rain_score,   'slope':         slope_score,
        'aspect':        aspect_score, 'soil':          soil_score,
        'precipWettest': pw_score,     'seasonality':   seas_score,
    }
    return overall.rename(name.replace(' ', '_')), scores_dict


# Mapping score_dict key -> PARAMETER_SETTINGS key (needed because key names differ).
_SCORE_TO_PARAM = {
    'alt':           'altitude',
    'frost':         'frost',
    'heat':          'maxTemp',
    'ph':            'pH',
    'rain':          'rainfall',
    'slope':         'slope',
    'aspect':        'aspect',
    'soil':          'soil',
    'precipWettest': 'precipWettest',
    'seasonality':   'precipSeasonality',
}

def calculate_limiting_factors(suitability, scores):
    """Returns (failure_count, limiting_factor_code).

    Disabled parameters (PARAMETER_SETTINGS[...]['enabled'] == False) are
    treated as never-failing and never-marginal here, so they don't appear
    in FailureCount or LimitingFactor outputs. This keeps the two rasters
    consistent with Suitability: if a parameter doesn't influence the
    suitability decision, it shouldn't be reported as a limit either.

    failure_count : per-pixel count of ENABLED parameters with score == 0 (0-N).
    limiting_factor : single code 1-15 - see legend at top of this cell.
    """
    s = scores

    def fail(key):
        if not PARAMETER_SETTINGS[_SCORE_TO_PARAM[key]]['enabled']:
            return ee.Image(0)
        return s[key].eq(0)

    def marg(key):
        if not PARAMETER_SETTINGS[_SCORE_TO_PARAM[key]]['enabled']:
            return ee.Image(0)
        return s[key].eq(1)

    # Failure count across enabled parameters only
    failure_count = (fail('alt').add(fail('frost')).add(fail('heat'))
                     .add(fail('ph')).add(fail('rain')).add(fail('slope'))
                     .add(fail('aspect')).add(fail('soil'))
                     .add(fail('precipWettest')).add(fail('seasonality')))

    # Theme-level fail / marginal counts (disabled params contribute 0)
    climate_fails = fail('frost').add(fail('heat')).add(fail('seasonality'))
    climate_margs = marg('frost').add(marg('heat')).add(marg('seasonality'))
    water_fails   = fail('rain').add(fail('precipWettest'))
    water_margs   = marg('rain').add(marg('precipWettest'))
    terrain_fails = fail('slope').add(fail('aspect'))
    terrain_margs = marg('slope').add(marg('aspect'))
    soil_fails    = fail('soil').add(fail('ph'))
    soil_margs    = marg('soil').add(marg('ph'))

    themes_failing = (climate_fails.gt(0).add(water_fails.gt(0))
                      .add(terrain_fails.gt(0)).add(soil_fails.gt(0)))
    single_theme   = themes_failing.eq(1)
    two_themes     = themes_failing.eq(2)

    lf = ee.Image(0)
    # Single-factor codes (1-8) — fail() is already 0 for disabled params,
    # so codes 3/4 stay 0 when rainfall/seasonality are disabled.
    lf = lf.where(fail('frost').And(single_theme),       1)
    lf = lf.where(fail('heat').And(single_theme),        2)
    lf = lf.where(fail('rain').And(single_theme),        3)
    lf = lf.where(fail('seasonality').And(single_theme), 4)
    lf = lf.where(fail('slope').And(single_theme),       5)
    lf = lf.where(fail('aspect').And(single_theme),      6)
    lf = lf.where(fail('soil').And(single_theme),        7)
    lf = lf.where(fail('ph').And(single_theme),          8)

    lf = lf.where(climate_fails.gt(0).And(water_fails.gt(0)).And(two_themes),    9)
    lf = lf.where(climate_fails.gt(0).And(terrain_fails.gt(0)).And(two_themes), 10)
    lf = lf.where(water_fails.gt(0).And(terrain_fails.gt(0)).And(two_themes),   11)
    lf = lf.where(soil_fails.gt(0).And(terrain_fails.gt(0)).And(two_themes),    12)
    lf = lf.where(climate_fails.gt(0).And(soil_fails.gt(0)).And(two_themes),    13)

    lf = lf.where(themes_failing.gte(3), 14)

    total_margs = climate_margs.add(water_margs).add(terrain_margs).add(soil_margs)
    lf = lf.where(suitability.eq(1).And(total_margs.gt(2)).And(failure_count.eq(0)), 15)

    return failure_count, lf

print('Scoring functions defined')


In [ ]:
# ============================================================
# CELL 7 - Submit export tasks
#
# For each species (alphabetical, optionally filtered by
# SPECIES_START_FROM in Cell 3): compute Suitability,
# FailureCount, LimitingFactor and submit 3 Export.image.toDrive
# tasks - UNLESS a task with the same name already exists in your
# GEE account in a useful state (COMPLETED, RUNNING, or READY).
# This makes the cell idempotent and resumable across sessions:
# if you've already exported some species, re-running just
# submits the missing ones.
# ============================================================

import time
import re

# Rerun guard - if you re-run this cell within the same session,
# fail loudly instead of submitting duplicates. To force a resubmit,
# manually run `submitted_tasks_suit = []` in a new cell.
if globals().get('submitted_tasks_suit'):
    raise RuntimeError(
        f'{len(submitted_tasks_suit)} tasks already submitted in this session - '
        f'clear `submitted_tasks_suit = []` manually if you really want to resubmit.'
    )

# ── Alphabetical species filter ────────────────────────────
species_list = sorted(SPECIES, key=lambda s: s[IDX['name']])
if SPECIES_START_FROM:
    cutoff = SPECIES_START_FROM.upper()
    species_list = [s for s in species_list if s[IDX['name']].upper() >= cutoff]

filter_note = f' (from "{SPECIES_START_FROM}" onwards)' if SPECIES_START_FROM else ''
print(f'Processing {len(species_list)} of {len(SPECIES)} species{filter_note}\n')

# ── Cache existing useful tasks (timestamp-stripped) ───────
USEFUL_STATES = {'COMPLETED', 'RUNNING', 'READY'}
_TS_PATTERN   = re.compile(r'_\d{8}_\d{4}$')

def _strip_ts(d):
    return _TS_PATTERN.sub('', d or '')

existing_no_ts = set()
for t in ee.batch.Task.list():
    st = t.status()
    if st.get('state') in USEFUL_STATES:
        existing_no_ts.add(_strip_ts(st.get('description')))

print(f'Found {len(existing_no_ts)} existing useful task(s) in your GEE account.')
print(f'Will SKIP exports whose name (timestamp-stripped) matches one of them.\n')

# ── Submit ─────────────────────────────────────────────────
submitted_tasks_suit = []
skipped_existing     = []

for species in species_list:
    name = species[IDX['name']]
    safe = name.replace(' ', '_')

    try:
        suitability, scores     = calculate_species_suitability(species)
        failure_count, limiting = calculate_limiting_factors(suitability, scores)

        layers = [
            ('Suitability',    suitability.updateMask(scope_mask).toByte()),
            ('FailureCount',   failure_count.updateMask(scope_mask).toByte()),
            ('LimitingFactor', limiting.updateMask(scope_mask).toByte()),
        ]

        for layer_name, img in layers:
            desc       = f'{layer_name}_{EXPORT_REGION_TAG}{safe}_{TIMESTAMP}'[:100]
            desc_no_ts = _strip_ts(desc)

            if desc_no_ts in existing_no_ts:
                skipped_existing.append(desc_no_ts)
                print(f'  = SKIP (exists)  {desc_no_ts}')
                continue

            task = ee.batch.Export.image.toDrive(
                image          = img,
                description    = desc,
                folder         = DRIVE_FOLDER,
                fileNamePrefix = desc,
                scale          = EXPORT_SCALE_M,
                crs            = 'EPSG:4326',
                region         = region_geom,
                maxPixels      = 1e13,
                fileFormat     = 'GeoTIFF',
            )
            task.start()
            submitted_tasks_suit.append(task)
            print(f'  + SUBMITTED      {desc}')
        time.sleep(1)

    except Exception as e:
        print(f'  x ERROR {name}: {e}')

# ── Summary ────────────────────────────────────────────────
print(f'\n{len(submitted_tasks_suit)} task(s) submitted to "{DRIVE_FOLDER}"')
print(f'{len(skipped_existing)} task(s) skipped (already existed in COMPLETED/RUNNING/READY state)')
if submitted_tasks_suit:
    print('Run Cell 8 to monitor.')
else:
    print('Nothing to do - all matching tasks already exist.')


In [ ]:
# ============================================================
# CELL 8 - Task status monitor (backend-aware, region-scoped)
# Queries GEE for every task whose description matches the
# suitability naming pattern for the CURRENT region.
# Re-run this cell any time to refresh.
# ============================================================

from collections import Counter

PATTERN_PREFIXES = (
    f'Suitability_{EXPORT_REGION_TAG}',
    f'FailureCount_{EXPORT_REGION_TAG}',
    f'LimitingFactor_{EXPORT_REGION_TAG}',
)

matching = []
for t in ee.batch.Task.list():
    status = t.status()
    desc   = status.get('description') or ''
    state  = status.get('state') or '?'
    if desc.startswith(PATTERN_PREFIXES):
        matching.append((desc, state))

if not matching:
    print(f'No suitability tasks found for region "{REGION}".')
else:
    matching.sort(key=lambda x: x[0])
    icons = {
        'COMPLETED':        '[OK]',
        'FAILED':           '[FAIL]',
        'RUNNING':          '[RUN]',
        'READY':            '[WAIT]',
        'CANCELLED':        '[CANC]',
        'CANCEL_REQUESTED': '[CANC]',
    }
    print(f'Status for {len(matching)} suitability tasks ({REGION}):\n')
    for desc, state in matching:
        print(f'  {icons.get(state, "[?]"):<8}  {state:<18}  {desc}')

    counts = Counter(state for _, state in matching)
    print('\nSummary:')
    for state, n in sorted(counts.items()):
        print(f'  {state:<18}  {n}')

    active = sum(n for s, n in counts.items() if s in ('READY', 'RUNNING'))
    if active == 0:
        print('\nAll tasks finished. Check Drive for the exported files.')
    else:
        print(f'\n{active} task(s) still active - re-run this cell to refresh.')
